# Step 5 — Model Training

Build sklearn pipelines, run 5-fold cross-validation, train all models on full training set.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import joblib

X_train = pd.read_csv('../../data/processed/X_train.csv', index_col=0)
y_train = pd.read_csv('../../data/processed/y_train.csv', index_col=0).squeeze()

print('X_train:', X_train.shape)
print('y_train:', y_train.shape)

## 5.1 Build Preprocessing Pipeline

In [ ]:
NUM_FEATURES = ['Age', 'Job', 'Credit amount', 'Duration', 'Credit_per_Duration']
CAT_FEATURES = ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose', 'Age_Group']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), NUM_FEATURES),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CAT_FEATURES)
])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':        RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':              XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
}

print('Preprocessor ready.')
print('Models:', list(models.keys()))

## 5.2 Cross-Validation (5-Fold Stratified)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print('=== 5-Fold Cross-Validation (ROC-AUC) ===')
for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    cv_results[name] = scores
    print(f'{name:25s} | Mean: {scores.mean():.4f} | Std: {scores.std():.4f} | Folds: {np.round(scores, 4)}')

In [ ]:
# CV results visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].boxplot(cv_results.values(), labels=cv_results.keys())
axes[0].set_title('CV ROC-AUC Distribution')
axes[0].set_ylabel('AUC Score')
axes[0].grid(axis='y', linestyle='--', alpha=0.7)

means = [v.mean() for v in cv_results.values()]
stds  = [v.std()  for v in cv_results.values()]
axes[1].barh(list(cv_results.keys()), means, xerr=stds,
             color='steelblue', edgecolor='black', capsize=5)
axes[1].set_title('Mean CV AUC with Std Dev')
axes[1].set_xlabel('ROC-AUC')
axes[1].set_xlim(0.5, 1.0)

plt.tight_layout()
plt.show()

## 5.3 Train All Models on Full Training Set

In [ ]:
trained_pipelines = {}

for name, model in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    print(f'{name}: trained ✓')

# Save all trained pipelines
joblib.dump(trained_pipelines, '../../model/all_pipelines.pkl')
print('\nAll pipelines saved to model/all_pipelines.pkl')